# 06b · Q1 : pourquoi les ventes ont chuté

**Projet 4 : Analyse des données de vente et retours** · Sprint 4 · Question Q1 (facteurs de performance des ventes)

**Deux notebooks pour Q1** : celui-ci explique la chute des ventes du côté de l'offre (nouveautés, gamme, produits vendus par magasin, stock dormant ou assortiment réduit, grands comptes). `06_q1_facteurs_rentabilite` mesure le poids de chaque facteur, la rentabilité après charges et chiffre les leviers, dont le renouvellement de l'assortiment recommandé ici.

**Point de départ** (`05b_charges`, section 7) : entre 2019 et 2021, les magasins perdent 48,6 % de leur chiffre d'affaires, alors que la boutique en ligne progresse et que le revendeur recule de 14 %. Leurs charges fixes n'ont pas suivi : c'est ce qui les fait passer en déficit. Ce notebook cherche **pourquoi les ventes des magasins ont chuté**, du côté de l'offre.

**La thèse de l'équipe** : ce ne sont pas les clients qui n'achètent plus, c'est le produit qui n'est plus là. Elle se décompose en deux hypothèses :

| Hypothèse | Ce qu'elle affirme | Comment on la teste |
|---|---|---|
| H1. Moins de nouveautés | Le catalogue se renouvelle moins, et ce sont les nouveautés qui faisaient les ventes | Lancements par année, chiffre d'affaires selon l'âge des produits, par canal |
| H2. Le produit n'est pas vendu dans le magasin | Un magasin vend de moins en moins de produits différents | Produits vendus par magasin, et pour chaque produit absent des ventes d'un magasin : est-il en rayon sans se vendre, ou absent du rayon ? |

Pour H2, un produit X qui ne se vend pas dans le magasin Y pendant l'année est dans l'un de deux états :

- **branche A, stock mort** : X est en rayon, mais ne se vend pas ;
- **branche B, assortiment réduit** : X n'est pas en rayon (retiré, jamais référencé, ou en rupture).

La table de stock ne permet pas de trancher directement (`05c_stock_objectifs`, section 3). On cherche donc, dans les ventes, les **indices** qui favorisent l'une ou l'autre branche, puis on montre que **les deux branches conduisent à la même recommandation**.

**Règle d'écriture des conclusions** : chaque hypothèse reçoit un verdict à la hauteur de sa preuve : *retenue* (mesurée), *indice* (compatible avec les données, sans preuve directe), *non soutenue* (les données ne montrent pas la trace qu'elle devrait laisser) ou *ouverte* (les données ne permettent pas de répondre). Jamais « fausse » sans test qui le démontre.

| Étape | Ce qu'on fait |
|---|---|
| 1. Chargement | Ventes, agrégats de l'offre, catalogue |
| 2. La chute à expliquer | Magasins, en ligne et revendeur : chiffre d'affaires, unités, prix moyen, uniformité |
| 3. Le glissement vers l'entrée de gamme | Gammes de prix, mêmes produits, même année de lancement, chronologie, contexte économique |
| 4. H1, les nouveautés | Lancements, chiffre d'affaires selon l'âge des produits, part de la chute |
| 5. H2, les produits vendus par magasin | Décomposition de la chute : moins de couples (magasin, produit), ou moins par couple ? |
| 6. H2, stock mort ou assortiment réduit ? | Quatre indices |
| 7. Les entreprises clientes en ligne | Départ des grands comptes, arrivée de nouveaux clients, part des nouveautés dans leurs achats |
| 8. Une seule stratégie | Renouveler l'assortiment ; ce que `06_q1_facteurs_rentabilite` chiffre |
| 9. Synthèse | Verdicts, limites, enregistrement |

**Lit dans** : `data/clean/fait_ventes_variables.parquet`, `data/clean/fait_stock.parquet`, `data/clean/fait_ventes_en_ligne.parquet`, `data/clean/dimensions/dim_produit.parquet`, `data/raw/contoso/Dim` (`DimCustomer`, `DimGeography`)

**Écrit dans** : `reports/figures` (figures `06b_*.png`) et `reports/livrables/L4_analyse/offre_tableaux.xlsx`

**Périmètre** : année de référence 2019, exercice 2021 (`config.ANNEE_REFERENCE`, `config.EXERCICE`). Les comparaisons entre magasins ne gardent que les **magasins ouverts toute l'année** (`05_variables`).

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

from src import config, viz

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

EXERCICE, REFERENCE = config.EXERCICE, config.ANNEE_REFERENCE
ANNEES = config.ANNEES_ETUDE
LIBELLES_TYPES = {"Store": "Magasins", "Online": "En ligne", "Reseller": "Revendeur"}
ORDRE_TYPES = ["Magasins", "En ligne", "Revendeur"]
print("Année de référence :", REFERENCE, "· exercice :", EXERCICE, "· années :", ANNEES)

# Registre des hypothèses : une ligne par test, avec son verdict
verdicts = []


def verdict(hypothese, test, resultat, conclusion):
    """Ajoute un test au registre des hypothèses et l'affiche.

    conclusion : "retenue", "indice", "non soutenue" ou "ouverte"
    """
    verdicts.append({"hypothèse": hypothese, "test": test, "résultat": resultat, "verdict": conclusion})
    print(f"[{conclusion:<12}] {hypothese} · {test} : {resultat}")


def millions(x):
    """Montant en millions de dollars, une décimale, virgule française."""
    return f"{x / 1e6:,.1f} M$".replace(",", " ").replace(".", ",")

## 1. Chargement

In [ ]:
ventes = pd.read_parquet(config.CLEAN / "fait_ventes_variables.parquet",
                         columns=["id_vente", "date", "annee", "mois", "id_boutique", "nom_boutique", "type_boutique", "pays",
                                  "id_produit", "categorie", "sous_categorie", "prix_unitaire", "quantite_vente", "ca_brut",
                                  "marge_nette", "date_ouverture", "grande_quantite", "severite_covid"])
dim_produit = pd.read_parquet(config.CLEAN_DIMENSIONS / "dim_produit.parquet")
stock = pd.read_parquet(config.CLEAN / "fait_stock.parquet", columns=["annee", "id_boutique", "id_produit"])

ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
lancement = dim_produit.set_index("id_produit").date_mise_en_vente.dt.year
ventes["annee_de_lancement"] = ventes.id_produit.map(lancement)

# Magasins ouverts toute l'année : ouverts au 1er janvier
ouverture = ventes.groupby("id_boutique").date_ouverture.first()
type_boutique = ventes.groupby("id_boutique").type_boutique.first()
magasins_complets = {a: sorted(ouverture[(ouverture < pd.Timestamp(a, 1, 2)) & (type_boutique == "Store")].index) for a in ANNEES}
ventes["magasin_complet"] = [i in magasins_complets[a] for i, a in zip(ventes.id_boutique, ventes.annee)]
magasins = ventes[ventes.magasin_complet]
print("Magasins ouverts toute l'année :", {a: len(m) for a, m in magasins_complets.items()})
print(f"Lignes : {len(ventes):,} · dont magasins ouverts toute l'année : {len(magasins):,}".replace(",", " "))

## 2. La chute à expliquer

Avant de chercher une cause, on décrit précisément ce qui baisse : le chiffre d'affaires, les unités, ou le prix moyen d'une unité ? Et la baisse est-elle concentrée (certains magasins, certains mois, certaines catégories) ou générale ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Chiffre d'affaires, unités, lignes et prix moyen d'une unité, par canal et par année
# ---------------------------------------------------------------------------
canal = ventes.groupby(["type", "annee"]).agg(ca_brut=("ca_brut", "sum"), unites=("quantite_vente", "sum"),
                                              lignes=("id_vente", "size"), produits_differents=("id_produit", "nunique"))
canal["prix moyen d'une unité"] = canal.ca_brut / canal.unites
canal["unités par ligne"] = canal.unites / canal.lignes
display(canal.loc[ORDRE_TYPES].style.format({"ca_brut": "{:,.0f}", "unites": "{:,.0f}", "lignes": "{:,.0f}", "produits_differents": "{:,.0f}",
                                             "prix moyen d'une unité": "{:,.2f}", "unités par ligne": "{:.1f}"}))

evolution = (canal.xs(EXERCICE, level="annee") / canal.xs(REFERENCE, level="annee") - 1).loc[ORDRE_TYPES]
print(f"Évolution de {REFERENCE} à {EXERCICE} :")
display(evolution[["ca_brut", "unites", "lignes", "prix moyen d'une unité"]].style.format("{:+.1%}"))

# ---------------------------------------------------------------------------
# 2. La baisse est-elle la même dans tous les magasins ?
# ---------------------------------------------------------------------------
complets_deux_annees = sorted(set(magasins_complets[REFERENCE]) & set(magasins_complets[EXERCICE]))
par_magasin = (ventes[ventes.id_boutique.isin(complets_deux_annees)]
               .pivot_table(index="nom_boutique", columns="annee", values="ca_brut", aggfunc="sum"))
par_magasin["évolution"] = par_magasin[EXERCICE] / par_magasin[REFERENCE] - 1
print(f"Évolution du chiffre d'affaires de {REFERENCE} à {EXERCICE}, magasins ouverts les deux années ({len(par_magasin)}) :")
display(par_magasin["évolution"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame().T.style.format(
    {c: "{:+.1%}" for c in ["mean", "min", "5%", "25%", "50%", "75%", "95%", "max"]} | {"count": "{:.0f}", "std": "{:.1%}"}))
print("Les magasins qui s'écartent le plus de la médiane :")
ecart = (par_magasin["évolution"] - par_magasin["évolution"].median()).abs().sort_values(ascending=False)
display(par_magasin.loc[ecart.index[:3]].style.format({REFERENCE: "{:,.0f}", EXERCICE: "{:,.0f}", "évolution": "{:+.1%}"}
                                                      | {a: "{:,.0f}" for a in ANNEES}))

# ---------------------------------------------------------------------------
# 3. ... tous les mois, toutes les catégories ?
# ---------------------------------------------------------------------------
mois = magasins.pivot_table(index="mois", columns="annee", values="ca_brut", aggfunc="sum")
print(f"Magasins : évolution mois par mois, {REFERENCE} → {EXERCICE} : de {(mois[EXERCICE] / mois[REFERENCE] - 1).min():+.0%} "
      f"à {(mois[EXERCICE] / mois[REFERENCE] - 1).max():+.0%}")
categories = ventes.pivot_table(index="categorie", columns=["type", "annee"], values="ca_brut", aggfunc="sum")
par_categorie = pd.DataFrame({t: categories[(t, EXERCICE)] / categories[(t, REFERENCE)] - 1 for t in ORDRE_TYPES})
par_categorie["écart magasins − en ligne (points)"] = (par_categorie["Magasins"] - par_categorie["En ligne"]) * 100
par_categorie["poids dans les magasins en " + str(REFERENCE)] = categories[("Magasins", REFERENCE)] / categories[("Magasins", REFERENCE)].sum()
print(f"Évolution du chiffre d'affaires par catégorie, {REFERENCE} → {EXERCICE} :")
display(par_categorie.sort_values("poids dans les magasins en " + str(REFERENCE), ascending=False).style.format(
    {"Magasins": "{:+.0%}", "En ligne": "{:+.0%}", "Revendeur": "{:+.0%}", "écart magasins − en ligne (points)": "{:+.0f}",
     "poids dans les magasins en " + str(REFERENCE): "{:.1%}"}))

**Lecture**

- **Le recul des magasins est un recul de volume.** De 2019 à 2021, le prix moyen d'une unité vendue baisse dans les trois canaux (−18 % en magasin, −33 % en ligne, −29 % chez le revendeur) : les clients achètent des produits moins chers partout. Ce qui sépare les canaux, ce sont les unités : −37,5 % en magasin, contre +59,7 % en ligne et +21,4 % chez le revendeur.
- **La baisse est générale.** Entre le 5e et le 95e centile, les magasins ouverts les deux années perdent entre 50,9 % et 56,2 % de leur chiffre d'affaires. Deux exceptions seulement : Moscow No.2 (+23,6 %) et Cambridge (−39,6 %). Chaque mois recule de 42 % à 58 %.
- **Toutes les catégories reculent plus en magasin qu'en ligne**, de 27 points (ordinateurs) à 91 points (télévision et vidéo) pour les catégories qui pèsent. Le recul n'est donc pas porté par une famille de produits.
- Une cause qui toucherait un magasin, une région, une saison ou une catégorie ne produirait pas cette uniformité. Ce qu'on cherche agit sur **l'ensemble du réseau de magasins**, et concerne d'abord le **nombre d'unités vendues**.

## 3. Le glissement vers l'entrée de gamme

Le prix catalogue d'un produit ne change jamais (`03_diagnostic`, section 11). Si le prix moyen d'une unité vendue baisse, c'est donc que les clients achètent **d'autres produits**, moins chers. On le vérifie de quatre façons :

1. par **gamme de prix** absolue, canal par canal ;
2. **à l'intérieur de chaque sous-catégorie** : les 20 % de produits les moins chers prennent-ils des unités aux 20 % les plus chers ?
3. sur les **mêmes produits** (vendus en 2019 et en 2021), et **à année de lancement égale**, pour écarter deux explications : le changement de catalogue et le vieillissement des produits ;
4. dans le **temps** et face à la **sévérité des restrictions COVID**, pays par pays.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Unités et chiffre d'affaires par gamme de prix, par canal
# ---------------------------------------------------------------------------
GAMMES = ["moins de 20 $", "20 à 100 $", "100 à 500 $", "plus de 500 $"]
ventes["gamme"] = pd.cut(ventes.prix_unitaire, [0, 20, 100, 500, 1e6], labels=GAMMES)
gammes = ventes.pivot_table(index=["type", "gamme"], columns="annee", values=["quantite_vente", "ca_brut"], aggfunc="sum", observed=True)
tableau_gammes = pd.concat({"unités vendues": gammes["quantite_vente"][[REFERENCE, EXERCICE]],
                            "chiffre d'affaires (M$)": gammes["ca_brut"][[REFERENCE, EXERCICE]] / 1e6}, axis=1).loc[ORDRE_TYPES]
print("Unités vendues et chiffre d'affaires par gamme de prix :")
display(tableau_gammes.style.format({c: ("{:,.0f}" if c[0] == "unités vendues" else "{:,.1f}") for c in tableau_gammes.columns}))

# ---------------------------------------------------------------------------
# 2. À l'intérieur de chaque sous-catégorie : les moins chers contre les plus chers
# ---------------------------------------------------------------------------
# Chaque produit est classé dans sa sous-catégorie selon son prix catalogue : 20 % les moins chers,
# milieu, 20 % les plus chers. On compare ainsi un téléviseur à des téléviseurs, pas à un câble.
POSITIONS = ["20 % les moins chers", "milieu", "20 % les plus chers"]


def positions_prix(produits):
    # Position de prix de chaque produit dans sa sous-catégorie (parmi les produits donnés)
    catalogue = dim_produit[dim_produit.id_produit.isin(produits)].copy()
    rang = catalogue.groupby("sous_categorie").prix_catalogue.rank(pct=True, method="first")
    return pd.Series(pd.cut(rang, [0, 0.2, 0.8, 1], labels=POSITIONS).values, index=catalogue.id_produit)


def part_des_unites(table, position):
    # Part des unités vendues de chaque position de prix, par canal et par année
    t = table.assign(position=table.id_produit.map(position))
    u = t.pivot_table(index=["type", "position"], columns="annee", values="quantite_vente", aggfunc="sum", observed=True)
    return (u / u.groupby(level="type").transform("sum")).loc[ORDRE_TYPES]


tous = part_des_unites(ventes, positions_prix(ventes.id_produit.unique()))
print("Part des unités vendues selon la position de prix dans la sous-catégorie (tous les produits) :")
display(tous.style.format("{:.1%}"))

# Combien de sous-catégories, en magasin, voient la part de leurs produits les moins chers augmenter ?
position_tous = positions_prix(ventes.id_produit.unique())
m = magasins.assign(position=magasins.id_produit.map(position_tous))
par_sc = m.pivot_table(index=["sous_categorie", "position"], columns="annee", values="quantite_vente", aggfunc="sum", observed=True, fill_value=0)
par_sc = par_sc / par_sc.groupby(level="sous_categorie").transform("sum")
bas_sc = par_sc.xs(POSITIONS[0], level="position")
print(f"Sous-catégories (magasins) où la part des 20 % les moins chers augmente de {REFERENCE} à {EXERCICE} :",
      int((bas_sc[EXERCICE] > bas_sc[REFERENCE]).sum()), "sur", len(bas_sc))

# ---------------------------------------------------------------------------
# 3. Mêmes produits, et même année de lancement
# ---------------------------------------------------------------------------
memes = sorted(set(ventes.loc[ventes.annee == REFERENCE, "id_produit"]) & set(ventes.loc[ventes.annee == EXERCICE, "id_produit"]))
position_memes = positions_prix(memes)
memes_produits = part_des_unites(ventes[ventes.id_produit.isin(memes)], position_memes)
print(f"Même mesure, sur les {len(memes)} produits vendus en {REFERENCE} et en {EXERCICE} (catalogue identique) :")
display(memes_produits.style.format("{:.1%}"))

v_memes = ventes[ventes.id_produit.isin(memes)].assign(position=lambda d: d.id_produit.map(position_memes))
cohortes = v_memes.pivot_table(index="annee_de_lancement", columns=["position", "annee"], values="quantite_vente", aggfunc="sum", observed=True)
rapport_cohortes = pd.DataFrame({pos: cohortes[(pos, EXERCICE)] / cohortes[(pos, REFERENCE)] for pos in POSITIONS})
print(f"Mêmes produits, par année de lancement : unités {EXERCICE} ÷ unités {REFERENCE}, tous canaux :")
display(rapport_cohortes.style.format("{:.2f}"))

# ---------------------------------------------------------------------------
# 4. Chronologie et COVID
# ---------------------------------------------------------------------------
m["trimestre"] = m.date.dt.to_period("Q")
chronologie = m.groupby("trimestre").apply(lambda d: d.loc[d.position == POSITIONS[0], "quantite_vente"].sum() / d.quantite_vente.sum(),
                                           include_groups=False).rename("part des 20 % les moins chers").to_frame()
chronologie["indice COVID moyen"] = m.groupby("trimestre").severite_covid.mean()
print("Magasins : part des unités des 20 % les moins chers, par trimestre :")
display(chronologie.style.format({"part des 20 % les moins chers": "{:.1%}", "indice COVID moyen": "{:.0f}"}))

# Pays par mois, en 2020 et 2021 : un pays aux restrictions plus fortes glisse-t-il plus vers l'entrée de gamme ?
annees_covid = config.ANNEES_ATTENDUES
pm = (m[m.annee.isin(annees_covid)].assign(mois_annee=lambda d: d.date.dt.to_period("M"))
      .groupby(["pays", "mois_annee"])
      .apply(lambda d: pd.Series({"part_bas": d.loc[d.position == POSITIONS[0], "quantite_vente"].sum() / d.quantite_vente.sum(),
                                  "severite": d.severite_covid.mean()}), include_groups=False))
correlation_brute = pm.part_bas.corr(pm.severite)
centre = pm - pm.groupby(level="mois_annee").transform("mean")          # à mois égal : on retire la moyenne du mois
correlation_mois_egal = centre.part_bas.corr(centre.severite)
print(f"Corrélation, pays × mois {annees_covid}, entre la sévérité COVID et la part des moins chers : "
      f"{correlation_brute:+.3f} · à mois égal : {correlation_mois_egal:+.3f}")

bas_m = tous.loc[("Magasins", POSITIONS[0])]
bas_e = tous.loc[("En ligne", POSITIONS[0])]
verdict("Entrée de gamme", "Les produits les moins chers de leur sous-catégorie gagnent-ils des unités ?",
        f"magasins {bas_m[REFERENCE]:.1%} → {bas_m[EXERCICE]:.1%}, en ligne {bas_e[REFERENCE]:.1%} → {bas_e[EXERCICE]:.1%}", "retenue")
verdict("Entrée de gamme", "Le glissement existe-t-il sur les mêmes produits et à année de lancement égale ?",
        f"oui : {memes_produits.loc[('Magasins', POSITIONS[0]), REFERENCE]:.1%} → {memes_produits.loc[('Magasins', POSITIONS[0]), EXERCICE]:.1%} en magasin, sur un catalogue identique",
        "retenue")
verdict("Entrée de gamme", "Le glissement est-il propre aux magasins ?", "non : identique dans les trois canaux", "retenue")
verdict("Entrée de gamme", "Suit-il la sévérité des restrictions COVID ?",
        f"non : corrélation {correlation_brute:+.2f}, {correlation_mois_egal:+.2f} à mois égal ; marches en janvier 2020 et janvier 2021", "non soutenue")
verdict("Entrée de gamme", "Arbitrage des ménages pendant la crise (consommation en baisse, épargne record, inflation fin 2021)",
        "compatible avec les chiffres officiels d'Eurostat ; non mesurable dans nos données", "ouverte")

**Lecture**

- **Les gammes de prix montrent deux canaux qui ne perdent pas les mêmes articles.** En magasin, la chute est concentrée sur les produits à plus de 100 $ : de 100 à 500 $, le chiffre d'affaires passe de 255,5 à 110,3 M$ ; au-delà de 500 $, de 245,6 à 140,9 M$. En ligne, ces mêmes gammes sont stables (de 244 à 246 M$), et la hausse des unités vient presque entièrement des articles à moins de 100 $ (+513 000 unités pour 15 M$ de chiffre d'affaires en plus).
- **À l'intérieur de chaque sous-catégorie, les clients descendent en gamme.** La part des unités des 20 % de produits les moins chers passe d'environ 20 % à 27 %, celle des 20 % les plus chers d'environ 19 % à 14 %. C'est le cas dans 25 sous-catégories sur 31 en magasin.
- **Ce n'est ni le catalogue ni le vieillissement des produits.** Sur les mêmes 1 815 produits, le glissement est encore plus net (de 15 % à 25 % pour les moins chers, de 24 % à 16 % pour les plus chers). Et à année de lancement égale, les moins chers progressent et les plus chers reculent : pour les produits lancés en 2019, les unités des moins chers sont multipliées par 1,89 et celles des plus chers par 0,53.
- **Le glissement est le même dans les trois canaux.** Il fait baisser le chiffre d'affaires partout, mais il n'explique pas l'écart entre magasins et boutique en ligne.
- **Il ne suit pas les restrictions COVID** : il avance par marches, chaque mois de janvier (premier trimestre 2020, avant la pandémie, puis premier trimestre 2021), et il n'est pas plus fort dans les pays aux restrictions plus sévères.

**Le contexte économique, d'après les chiffres officiels.** Ce que nos données ne peuvent pas mesurer, les statistiques européennes l'éclairent :

- en 2020, la consommation des ménages de l'Union européenne recule d'environ 5 % par habitant, et le taux d'épargne atteint un record : 20 % du revenu disponible dans la zone euro au 4e trimestre 2020, sept points de plus qu'un an plus tôt ([Eurostat](https://ec.europa.eu/eurostat/statistics-explained/SEPDF/cache/89580.pdf)) ;
- le pouvoir d'achat mesuré (revenu disponible corrigé de l'inflation) est soutenu par les aides publiques en 2020 et 2021 ([Eurostat](https://ec.europa.eu/eurostat/documents/2995521/14497754/2-28042022-AP-EN.pdf/3e81dc37-d82f-2a83-08be-2bc744f526da)) ;
- mais cette mesure ne tient pas compte des dépenses contraintes nouvelles (télétravail, santé, logement), et l'inflation revient fortement fin 2021 : 5,0 % sur un an en décembre 2021 dans la zone euro, contre −0,3 % un an plus tôt ([Eurostat](https://ec.europa.eu/eurostat/documents/2995521/14176359/2-20012022-AP-EN.pdf/)).

**Verdict** : le glissement vers l'entrée de gamme est *retenu*, commun aux trois canaux. Son explication par les restrictions COVID est *non soutenue*. Un arbitrage des ménages pendant la crise (moins de dépenses, épargne de précaution, puis retour de l'inflation) est *compatible* avec les chiffres officiels, mais *ouvert* : nos données ne contiennent ni revenus ni moyens de paiement. *Réserve de chronologie* : la première marche apparaît dès janvier 2020, avant la pandémie.

## 4. H1 : moins de nouveautés

**Le fait** : le catalogue se renouvelle moins. On le chiffre, puis on mesure ce qu'il pèse dans la chute des magasins, en séparant le chiffre d'affaires des produits **lancés dans l'année** de celui des produits **plus anciens**.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Lancements par année (catalogue), et date du dernier lancement
# ---------------------------------------------------------------------------
vendus = dim_produit[dim_produit.id_produit.isin(ventes.id_produit.unique())].copy()
vendus["annee_de_lancement"] = vendus.date_mise_en_vente.dt.year
lancements = vendus.groupby("annee_de_lancement").agg(produits_lances=("id_produit", "size"),
                                                      dernier_lancement=("date_mise_en_vente", "max"))
display(lancements.style.format({"dernier_lancement": "{:%Y-%m-%d}"}))
variation_lancements = lancements.loc[EXERCICE, "produits_lances"] / lancements.loc[REFERENCE, "produits_lances"] - 1

# ---------------------------------------------------------------------------
# 2. Chiffre d'affaires selon l'âge du produit, par canal et par année
# ---------------------------------------------------------------------------
# Âge = année de vente − année de lancement : 0 = nouveauté de l'année, 1 = lancée l'année précédente, 2 et plus = plus ancienne
ventes["age"] = np.select([ventes.annee == ventes.annee_de_lancement, ventes.annee - ventes.annee_de_lancement == 1],
                          ["nouveauté de l'année", "lancé l'année précédente"], default="plus ancien")
# Les quelques ventes datées avant le lancement (05c, section 4) sont comptées comme nouveautés
ventes.loc[ventes.annee < ventes.annee_de_lancement, "age"] = "nouveauté de l'année"
ORDRE_AGES = ["nouveauté de l'année", "lancé l'année précédente", "plus ancien"]
age = ventes.pivot_table(index=["type", "age"], columns="annee", values="ca_brut", aggfunc="sum", fill_value=0)
age = age.reindex(pd.MultiIndex.from_product([ORDRE_TYPES, ORDRE_AGES]))
age["variation"] = age[EXERCICE] - age[REFERENCE]
age["évolution"] = age[EXERCICE] / age[REFERENCE] - 1
print("Chiffre d'affaires selon l'âge du produit (dollars) :")
display(age.style.format({a: "{:,.0f}" for a in ANNEES} | {"variation": "{:+,.0f}", "évolution": "{:+.0%}"}))

# Décomposition comptable de la variation des magasins selon l'âge des produits
part_magasins = age.loc["Magasins", "variation"] / age.loc["Magasins", "variation"].sum()
print(f"Magasins : part de la variation {REFERENCE} → {EXERCICE} portée par chaque âge (décomposition comptable) :")
display(part_magasins.to_frame("part").style.format("{:+.0%}"))

# ---------------------------------------------------------------------------
# 3. Les nouveautés arrivent-elles en magasin aussi bien qu'en ligne ?
# ---------------------------------------------------------------------------
# Rapport magasins ÷ en ligne du chiffre d'affaires, par âge du produit : si les nouveautés
# arrivaient moins bien en magasin, ce rapport serait plus bas pour elles que pour les produits anciens.
rapport = pd.DataFrame({a: age.loc["Magasins", a] / age.loc["En ligne", a] for a in ANNEES})
print("Chiffre d'affaires des magasins ÷ chiffre d'affaires en ligne, par âge du produit :")
display(rapport.loc[ORDRE_AGES].style.format("{:.2f}"))

# Part des magasins qui vendent un produit dans l'année, selon l'âge du produit
diffusion_age = (ventes[ventes.magasin_complet].groupby(["annee", "age", "id_produit"]).id_boutique.nunique().rename("magasins").reset_index())
diffusion_age["part des magasins"] = diffusion_age.magasins / diffusion_age.annee.map({a: len(m) for a, m in magasins_complets.items()})
print("Part des magasins (ouverts toute l'année) qui vendent un produit, selon son âge :")
display(diffusion_age.pivot_table(index="age", columns="annee", values="part des magasins", aggfunc="mean").loc[ORDRE_AGES].style.format("{:.0%}"))

v_nouv = age.loc[("Magasins", "nouveauté de l'année"), "variation"]
v_tot = age.loc["Magasins", "variation"].sum()
verdict("H1 nouveautés", "Lancements par année",
        f"{lancements.loc[REFERENCE, 'produits_lances']} en {REFERENCE}, {lancements.loc[EXERCICE, 'produits_lances']} en {EXERCICE} ({variation_lancements:+.0%})",
        "retenue")
verdict("H1 nouveautés", f"Part de la chute des magasins {REFERENCE} → {EXERCICE} due aux nouveautés de l'année",
        f"{millions(v_nouv)} sur {millions(v_tot)} ({v_nouv / v_tot:.0%})", "retenue")
v_nouv_en_ligne = age.loc[("En ligne", "nouveauté de l'année"), "variation"]
verdict("H1 nouveautés", "Les nouveautés reculent-elles aussi en ligne ?", f"oui, {millions(v_nouv_en_ligne)} en ligne", "retenue")
verdict("H1 nouveautés", "Les produits plus anciens compensent-ils, comme en ligne ?",
        f"non : {age.loc[('Magasins', 'plus ancien'), 'évolution']:+.0%} en magasin contre {age.loc[('En ligne', 'plus ancien'), 'évolution']:+.0%} en ligne",
        "ouverte")

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : chiffre d'affaires selon l'âge du produit, magasins et en ligne
# ---------------------------------------------------------------------------
couleurs_age = {"nouveauté de l'année": viz.ACCENT, "lancé l'année précédente": viz.CATEGORIES[2], "plus ancien": viz.PRINCIPALE}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, t in zip(axes, ["Magasins", "En ligne"]):
    bas = np.zeros(len(ANNEES))
    for a in ORDRE_AGES:
        valeurs = age.loc[(t, a), ANNEES].values.astype(float) / 1e6
        ax.bar([str(x) for x in ANNEES], valeurs, bottom=bas, color=couleurs_age[a], label=a, width=0.6, edgecolor="white")
        for x, (b, v) in enumerate(zip(bas, valeurs)):
            if v > 25:
                ax.text(x, b + v / 2, f"{v:,.0f}".replace(",", " "), ha="center", va="center", color="white", fontweight="bold")
        bas += valeurs
    for x, total in enumerate(bas):
        ax.text(x, total + 6, f"{total:,.0f} M$".replace(",", " "), ha="center", va="bottom", fontweight="bold", color=viz.FONCE)
    ax.set_title(t, fontsize=12, color=viz.FONCE, fontweight="bold")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Chiffre d'affaires (millions de dollars)")
axes[0].set_ylim(0, 600)
axes[1].legend(loc="upper right", frameon=False)
fig.suptitle("Les nouveautés reculent partout ; en ligne, les produits anciens compensent bien plus qu'en magasin",
             x=0.01, y=0.99, ha="left", fontsize=13, fontweight="bold", color=viz.FONCE)
fig.text(0.01, 0.925, "Chiffre d'affaires brut selon l'âge du produit (année de vente − année de lancement), en millions de dollars", ha="left", color=viz.GRIS)
fig.tight_layout(rect=(0, 0, 1, 0.93))
print("Figure enregistrée :", viz.enregistrer(fig, "06b_nouveautes"))
plt.show()

**Lecture**

- **Le fait est confirmé.** Le catalogue se renouvelle moins : 552 produits lancés en 2019, 429 en 2020, 272 en 2021 (−51 %). Le dernier lancement date du 21 juillet 2021, il n'y en a plus au second semestre.
- **C'est la plus grosse part de la chute des magasins.** Les nouveautés de l'année rapportaient 201,9 M$ aux magasins en 2019, et 28,0 M$ en 2021 (−86 %). Dans une décomposition comptable, elles portent 69 % de la baisse, et les produits lancés l'année précédente 51 %. Les produits plus anciens compensent une partie (+35 %, soit −20 % de la baisse).
- **Mais le recul des nouveautés n'est pas propre aux magasins.** En ligne aussi, les nouveautés de l'année passent de 97,6 M$ à 34,7 M$ (−64 %). La différence : en ligne, les produits plus anciens progressent de 158 % et compensent largement, en magasin de 35 % seulement.
- **Les magasins perdent du terrain sur tous les âges de produits, dans la même proportion.** Le rapport magasins ÷ en ligne passe de 2,07 à 0,81 pour les nouveautés, de 1,97 à 1,03 pour les produits anciens. Si les nouveautés arrivaient moins bien en magasin, ce rapport baisserait surtout pour elles. Ce n'est pas le cas.
- *Attention à la diffusion par âge* : une nouveauté est vendue dans moins de magasins qu'un produit ancien (35 % contre 42 % en 2021), mais une partie s'explique par sa date de lancement. Un produit lancé en juillet n'a que cinq mois pour se vendre.

**Verdict H1** : *retenue* comme fait et comme principale composante comptable de la chute des magasins. Elle n'explique pas, à elle seule, l'écart entre magasins et boutique en ligne : les deux canaux perdent leurs nouveautés, mais seule la boutique en ligne compense avec le reste du catalogue. Pourquoi : *ouverte*.

## 5. H2 : les produits vendus par magasin

Le chiffre d'affaires des magasins est la somme, sur chaque couple (magasin, produit) qui a au moins une vente dans l'année, de ce que ce couple vend. Il peut donc baisser de deux façons :

- **moins de couples** : chaque magasin vend moins de produits différents ;
- **moins par couple** : un produit vendu dans un magasin y rapporte moins.

On sépare les deux effets, d'abord pour l'ensemble des magasins, puis sur un groupe de produits suivis les trois années, en comparant avec la boutique en ligne.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Décomposition : couples vendus × chiffre d'affaires par couple
# ---------------------------------------------------------------------------
couples = magasins.groupby(["annee", "id_boutique", "id_produit"]).agg(ca=("ca_brut", "sum"), lignes=("id_vente", "size")).reset_index()
decomposition = couples.groupby("annee").agg(couples_vendus=("ca", "size"), ca_brut=("ca", "sum"), lignes=("lignes", "sum"))
decomposition["magasins"] = pd.Series({a: len(m) for a, m in magasins_complets.items()})
decomposition["produits différents vendus par magasin"] = decomposition.couples_vendus / decomposition.magasins
decomposition["ca par couple"] = decomposition.ca_brut / decomposition.couples_vendus
decomposition["lignes par couple"] = decomposition.lignes / decomposition.couples_vendus
display(decomposition.style.format({"couples_vendus": "{:,.0f}", "ca_brut": "{:,.0f}", "lignes": "{:,.0f}",
                                    "produits différents vendus par magasin": "{:,.0f}", "ca par couple": "{:,.0f}", "lignes par couple": "{:.2f}"}))

# Part de chaque effet, sur l'échelle logarithmique (les deux parts s'additionnent exactement)
d_ca = np.log(decomposition.loc[EXERCICE, "ca_brut"] / decomposition.loc[REFERENCE, "ca_brut"])
d_couples = np.log(decomposition.loc[EXERCICE, "couples_vendus"] / decomposition.loc[REFERENCE, "couples_vendus"])
print(f"De {REFERENCE} à {EXERCICE} : chiffre d'affaires {np.exp(d_ca) - 1:+.1%}, dont "
      f"{d_couples / d_ca:.0%} par moins de couples vendus et {(d_ca - d_couples) / d_ca:.0%} par moins de chiffre d'affaires par couple")

# ---------------------------------------------------------------------------
# 2. Les mêmes produits, en ligne et en magasin
# ---------------------------------------------------------------------------
# On garde les produits vendus en ligne chacune des trois années : leur demande existe sur toute la période.
en_ligne = ventes[ventes.type == "En ligne"]
annees_en_ligne = en_ligne.groupby("id_produit").annee.nunique()
suivis = annees_en_ligne[annees_en_ligne == len(ANNEES)].index
print(f"\nProduits vendus en ligne les {len(ANNEES)} années : {len(suivis)}")
cohorte = pd.DataFrame({
    "ca en ligne": en_ligne[en_ligne.id_produit.isin(suivis)].groupby("annee").ca_brut.sum(),
    "ca en magasin": magasins[magasins.id_produit.isin(suivis)].groupby("annee").ca_brut.sum(),
    "couples (magasin, produit) vendus": couples[couples.id_produit.isin(suivis)].groupby("annee").size(),
})
cohorte["magasins qui vendent un produit (moyenne)"] = cohorte["couples (magasin, produit) vendus"] / len(suivis)
cohorte["part des magasins"] = cohorte["magasins qui vendent un produit (moyenne)"] / decomposition.magasins
cohorte["ca par magasin qui vend le produit"] = cohorte["ca en magasin"] / cohorte["couples (magasin, produit) vendus"]
display(cohorte.style.format({"ca en ligne": "{:,.0f}", "ca en magasin": "{:,.0f}", "couples (magasin, produit) vendus": "{:,.0f}",
                              "magasins qui vendent un produit (moyenne)": "{:.1f}", "part des magasins": "{:.0%}",
                              "ca par magasin qui vend le produit": "{:,.0f}"}))
evol_cohorte = cohorte.loc[EXERCICE] / cohorte.loc[REFERENCE] - 1
print(f"Mêmes produits, {REFERENCE} → {EXERCICE} : en ligne {evol_cohorte['ca en ligne']:+.0%} · en magasin {evol_cohorte['ca en magasin']:+.0%}"
      f" · par magasin qui vend le produit {evol_cohorte['ca par magasin qui vend le produit']:+.0%}"
      f" · magasins qui le vendent {evol_cohorte['magasins qui vendent un produit (moyenne)']:+.0%}")

verdict("H2 produits par magasin", "Un magasin vend-il moins de produits différents dans l'année ?",
        f"{decomposition.loc[REFERENCE, 'produits différents vendus par magasin']:,.0f} en {REFERENCE}".replace(",", " ")
        + f" · {decomposition.loc[EXERCICE, 'produits différents vendus par magasin']:,.0f} en {EXERCICE}".replace(",", " "), "retenue")
verdict("H2 produits par magasin", "Part de la chute due aux couples (magasin, produit) qui ne vendent plus",
        f"{d_couples / d_ca:.0%}", "retenue")
verdict("H2 produits par magasin", "Là où il se vend, un produit recule-t-il comme en ligne ?",
        f"{evol_cohorte['ca par magasin qui vend le produit']:+.0%} par magasin qui le vend, {evol_cohorte['ca en ligne']:+.0%} en ligne", "retenue")

**Lecture**

- **Un magasin vend un tiers de produits différents en moins** : 1 508 en moyenne en 2019, 979 en 2021. Le nombre de couples (magasin, produit) qui vendent passe de 72 395 à 51 899.
- **La chute se partage à parts presque égales** : 51 % viennent des couples qui ne vendent plus, 49 % de ce que rapporte un couple qui vend encore (7 042 $ par an en 2019, 5 109 $ en 2021).
- **Les mêmes produits, suivis en ligne et en magasin**, montrent d'où vient l'écart. Sur les 1 813 produits vendus en ligne les trois années, le chiffre d'affaires en ligne recule de 26 % et celui des magasins de 62 %. Dans un magasin qui vend le produit, la baisse (−28 %) est la même qu'en ligne. **Ce qui est propre aux magasins, c'est que le produit s'y vend dans deux fois moins de magasins** : 39,8 en 2019, 21,1 en 2021.

**Verdict** : la réduction du nombre de produits vendus par magasin est *retenue* comme fait, et c'est elle qui fait l'écart entre magasins et boutique en ligne. Reste à savoir ce que cache un produit non vendu dans un magasin : c'est l'objet de la section 6.

## 6. Un produit non vendu dans un magasin : stock mort ou assortiment réduit ?

On ne voit pas les rayons, seulement les ventes. Mais les deux branches ne laissent pas les mêmes traces dans les ventes :

| Indice | Si c'est du stock mort (branche A) | Si c'est un assortiment réduit (branche B) |
|---|---|---|
| 1. Probabilité qu'un produit présent ne se vende pas de l'année | Élevée quand le produit se vend rarement : l'absence de vente s'explique par la rotation | Les absences dépassent ce que la rotation explique |
| 2. Répartition des ventes d'un produit entre magasins | Uniforme : chaque magasin a le produit | Concentrée : certains magasins ont le produit, d'autres non |
| 3. Répartition entre pays | Uniforme | Concentrée si les retraits ou les ruptures sont régionaux |
| 4. Couple sans vente une année : se vend-il l'année suivante ? | Oui, aussi souvent que la rotation le prévoit | Rarement : un produit retiré ne revient pas tout seul |

**Comment on calcule ce que « la rotation explique ».** Si un produit est présent dans tous les magasins et qu'il fait en moyenne λ lignes de vente par magasin et par an, le nombre de lignes d'un magasin suit une loi de Poisson de moyenne λ (des événements rares et indépendants). La probabilité qu'un magasin n'enregistre aucune vente de l'année vaut alors e^(−λ). Exemple : avec λ = 0,6, un magasin sur deux (e^(−0,6) = 55 %) ne vend pas le produit de l'année, alors qu'il l'a en rayon.

In [ ]:
# ---------------------------------------------------------------------------
# Lignes de vente par (produit, magasin) et par année, zéros compris
# ---------------------------------------------------------------------------
grilles = []
for a in ANNEES:
    m = magasins[magasins.annee == a]
    k = m.groupby(["id_produit", "id_boutique"]).size()
    grille = k.reindex(pd.MultiIndex.from_product([m.id_produit.unique(), magasins_complets[a]], names=["id_produit", "id_boutique"]),
                       fill_value=0).rename("lignes").reset_index()
    grille["annee"] = a
    grilles.append(grille)
grille = pd.concat(grilles, ignore_index=True)
grille["lambda"] = grille.groupby(["annee", "id_produit"]).lignes.transform("mean")
grille["vendu"] = grille.lignes > 0
pays = ventes.groupby("id_boutique").pays.first()
grille["pays"] = grille.id_boutique.map(pays)

# ---------------------------------------------------------------------------
# Indice 1 : part des magasins qui vendent un produit, observée et prévue par la rotation
# ---------------------------------------------------------------------------
par_produit = grille.groupby(["annee", "id_produit"]).agg(lam=("lambda", "first"), observee=("vendu", "mean"))
par_produit["prevue"] = 1 - np.exp(-par_produit.lam)
indice1 = par_produit.groupby("annee").agg(produits=("lam", "size"), lignes_par_magasin_lambda=("lam", "mean"),
                                           part_observee=("observee", "mean"), part_prevue=("prevue", "mean"))
indice1["couples sans vente (observé)"] = 1 - indice1.part_observee
print("Indice 1 · Part des magasins qui vendent un produit dans l'année :")
display(indice1.style.format({"produits": "{:,.0f}", "lignes_par_magasin_lambda": "{:.2f}", "part_observee": "{:.1%}",
                              "part_prevue": "{:.1%}", "couples sans vente (observé)": "{:.1%}"}))
par_produit["tranche"] = par_produit.groupby("annee").lam.transform(lambda s: pd.qcut(s, 5, labels=["1 (rotation la plus faible)", "2", "3", "4", "5 (la plus forte)"]))
print("Même comparaison par tranche de rotation (cinquièmes des produits) :")
display(par_produit.pivot_table(index="tranche", columns="annee", values=["observee", "prevue"], aggfunc="mean", observed=True)
        .style.format("{:.1%}"))

# ---------------------------------------------------------------------------
# Indice 2 : dispersion des lignes d'un produit entre magasins
# ---------------------------------------------------------------------------
# Variance ÷ moyenne du nombre de lignes par magasin, produit par produit.
#   = 1 : répartition au hasard entre des magasins qui ont TOUS le produit (loi de Poisson)
#   > 1 : répartition concentrée, ce que donnerait un produit proposé dans une partie des magasins seulement.
# Repère : si un produit n'était proposé que dans une part p des magasins, avec le même nombre moyen de lignes λ,
# l'indice vaudrait 1 + λ × (1 − p) / p.
dispersion = grille.groupby(["annee", "id_produit"]).lignes.agg(["mean", "var"])
dispersion["indice"] = dispersion["var"] / dispersion["mean"]
indice2 = dispersion.groupby("annee").indice.median().rename("indice médian observé").to_frame()
for p in (0.8, 0.6):
    indice2[f"si proposé dans {p:.0%} des magasins"] = 1 + indice1.lignes_par_magasin_lambda * (1 - p) / p
print("Indice 2 · Variance ÷ moyenne des lignes d'un produit entre magasins (médiane des produits) :")
display(indice2.style.format("{:.2f}"))

# ---------------------------------------------------------------------------
# Indice 3 : répartition entre pays
# ---------------------------------------------------------------------------
# Pour chaque produit et chaque pays : lignes observées contre lignes attendues si le produit se vendait
# au même rythme dans tous les magasins. Statistique du khi-deux divisée par ses degrés de liberté :
# proche de 1 = pas d'effet pays ; nettement au-dessus de 1 = ventes concentrées dans certains pays.
indice3 = {}
for a, g in grille.groupby("annee"):
    c = g.groupby(["id_produit", "pays"]).agg(observees=("lignes", "sum"), magasins=("lignes", "size"), lam=("lambda", "first"))
    c["attendues"] = c.magasins * c.lam
    c = c[c.attendues > 0]
    khi2 = ((c.observees - c.attendues) ** 2 / c.attendues).sum()
    ddl = len(c) - c.index.get_level_values("id_produit").nunique()
    indice3[a] = {"pays": g.pays.nunique(), "khi2 ÷ degrés de liberté": khi2 / ddl}
indice3 = pd.DataFrame(indice3).T
print("Indice 3 · Concentration des ventes d'un produit par pays :")
display(indice3.style.format({"pays": "{:.0f}", "khi2 ÷ degrés de liberté": "{:.3f}"}))

# ---------------------------------------------------------------------------
# Indice 4 : un couple sans vente une année se vend-il l'année suivante ?
# ---------------------------------------------------------------------------
indice4 = []
cle = ["id_produit", "id_boutique"]
for a, suivante in zip(ANNEES[:-1], ANNEES[1:]):
    avant = grille[grille.annee == a].set_index(cle)
    apres = grille[grille.annee == suivante].set_index(cle)
    communs = avant.index.intersection(apres.index)
    sans_vente = avant.loc[communs][~avant.loc[communs, "vendu"]]
    suite = apres.loc[sans_vente.index]
    indice4.append({"couples sans vente en": a, "couples": len(sans_vente),
                    f"part vendue l'année suivante (observée)": suite.vendu.mean(),
                    f"part prévue si le produit est resté en rayon": (1 - np.exp(-suite["lambda"])).mean()})
indice4 = pd.DataFrame(indice4).set_index("couples sans vente en")
print("Indice 4 · Couples (magasin, produit) sans vente une année, présents dans les deux années :")
display(indice4.style.format({"couples": "{:,.0f}", "part vendue l'année suivante (observée)": "{:.1%}",
                              "part prévue si le produit est resté en rayon": "{:.1%}"}))

# ---------------------------------------------------------------------------
# Ce que la table de stock ajoute (avec sa limite)
# ---------------------------------------------------------------------------
releves = stock.groupby(["annee", "id_boutique", "id_produit"]).size().rename("releves").reset_index()
avec_stock = grille.merge(releves, on=["annee", "id_boutique", "id_produit"], how="inner")
stock_sans_vente = avec_stock.groupby("annee").apply(lambda g: pd.Series({
    "couples avec un relevé de stock": len(g), "dont sans vente (observé)": (~g.vendu).mean(),
    "prévu par la rotation": np.exp(-g["lambda"]).mean()}), include_groups=False)
print("Couples (magasin, produit) qui ont au moins un relevé de stock dans l'année :")
display(stock_sans_vente.style.format({"couples avec un relevé de stock": "{:,.0f}", "dont sans vente (observé)": "{:.1%}",
                                       "prévu par la rotation": "{:.1%}"}))

ex = indice1.loc[EXERCICE]
verdict("H2 branche A (stock mort)", f"Indice 1 · couples sans vente en {EXERCICE} expliqués par la rotation",
        f"observé {1 - ex.part_observee:.1%}, prévu {1 - ex.part_prevue:.1%} (λ = {ex.lignes_par_magasin_lambda:.2f} ligne par magasin et par an)", "indice")
verdict("H2 branche B (assortiment réduit)", "Indice 2 · ventes d'un produit concentrées sur certains magasins",
        f"indice {indice2.loc[EXERCICE, 'indice médian observé']:.2f} (1 = aucune concentration ; "
        f"{indice2.loc[EXERCICE, 'si proposé dans 60% des magasins']:.2f} si proposé dans 60 % des magasins)", "non soutenue")
verdict("H2 branche B (assortiment réduit)", "Indice 3 · retraits ou ruptures régionaux",
        f"khi2 ÷ ddl = {indice3.loc[EXERCICE, 'khi2 ÷ degrés de liberté']:.2f}", "non soutenue")
verdict("H2 branche A (stock mort)", "Indice 4 · couples sans vente qui se vendent l'année suivante",
        f"{indice4.iloc[-1, 1]:.1%} observé, {indice4.iloc[-1, 2]:.1%} prévu si le produit est resté en rayon", "indice")
verdict("H2", "Proportion exacte de stock mort et d'assortiment réduit", "non mesurable : pas d'inventaire complet", "ouverte")

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : part des magasins qui vendent un produit, observée et prévue par la rotation
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 5.2))
couleurs_annees = dict(zip(ANNEES, [viz.PRINCIPALE, viz.CATEGORIES[2], viz.ACCENT]))
x_theorique = np.linspace(0, par_produit.lam.quantile(0.99), 200)
ax.plot(x_theorique, 1 - np.exp(-x_theorique), color=viz.FONCE, linewidth=2, label="Prévu si le produit est dans tous les magasins")
for a in ANNEES:
    p = par_produit.xs(a, level="annee")
    # Points regroupés en 20 tranches de rotation pour rester lisibles
    tranches = pd.qcut(p.lam, 20, duplicates="drop")
    moyenne = p.groupby(tranches, observed=True)[["lam", "observee"]].mean()
    ax.scatter(moyenne.lam, moyenne.observee, s=36, color=couleurs_annees[a], label=f"Observé en {a}", zorder=3)
ax.set_xlabel("Lignes de vente du produit par magasin et par an (λ)")
ax.set_ylabel("Part des magasins qui vendent le produit")
viz.format_pourcent(ax, axe="y")
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right", frameon=False)
viz.titre(ax, "Les absences de vente suivent la rotation des produits",
          "Chaque point : 5 % des produits de l'année · courbe : 1 − e^(−λ), produit présent partout")
print("Figure enregistrée :", viz.enregistrer(fig, "06b_rotation"))
plt.show()

**Lecture**

- **Indice 1, la rotation.** En 2021, un produit fait en moyenne 0,57 ligne de vente par magasin et par an, contre 2,49 en 2019. Même présent dans tous les magasins, il resterait invendu dans 58,6 % d'entre eux : c'est ce qu'on observe (58,9 %). La correspondance tient pour chaque tranche de rotation et chaque année, à un point près. Seul le cinquième des produits qui tournent le plus en 2021 s'en écarte un peu (54,0 % de magasins vendeurs contre 55,1 % prévus, et un point isolé sur le graphique).
- **Indice 2, la concentration entre magasins.** L'indice vaut 1,0 les trois années. Si un produit n'était proposé que dans 80 % des magasins, il vaudrait 1,14 en 2021, et 1,38 à 60 %. *Limite* : plus la rotation est faible, plus l'écart à détecter est petit. Le test est donc moins puissant en 2021 qu'en 2019 (1,62 et 2,66).
- **Indice 3, les pays.** Les ventes d'un produit ne sont pas concentrées dans certains pays (khi² ÷ degrés de liberté de 0,99 à 1,02). Des retraits ou des ruptures régionaux laisseraient cette trace.
- **Indice 4, le retour.** Parmi les couples sans vente en 2020, 41,7 % se vendent en 2021. C'est ce que prévoit la rotation si le produit était resté en rayon (41,9 %). Un produit retiré ne reviendrait pas aussi souvent.
- **La table de stock ne départage pas.** Les relevés tombent presque toujours là où il y a des ventes : 11 % seulement des couples relevés n'ont aucune vente, contre 40 à 56 % prévus par la rotation. Elle ne permet donc pas de compter le stock mort.

**Ce que disent ces indices**

| Branche | Verdict | Pourquoi |
|---|---|---|
| A. Stock mort (le produit est en rayon et ne se vend pas) | *Indice* favorable | Les absences de vente correspondent exactement à ce que donne une rotation trop faible, et les couples sans vente reviennent au rythme prévu |
| B. Assortiment réduit (le produit n'est pas en rayon) | *Non soutenue* | Aucune concentration par magasin ni par pays. Elle n'est pas exclue pour autant : un retrait réparti au hasard sur le réseau ne laisserait pas de trace dans ces tests |
| Proportion exacte de chaque branche | *Ouverte* | Pas d'inventaire complet |

**Lecture métier** : de 2019 à 2021, les magasins vendent moins d'unités, alors que le nombre de produits qu'ils vendent ensemble augmente (1 729 à 2 384). Chaque produit tourne donc de moins en moins : moins d'une ligne de vente par magasin et par an en 2021. L'assortiment vieillit (les produits de deux ans et plus passent de 28 % à 73 % du chiffre d'affaires des magasins) et une part croissante des références dort en rayon.

## 7. Les entreprises clientes de la boutique en ligne

La table des ventes en ligne détaillées est la seule qui identifie les clients (`05c_stock_objectifs`, partie C). Elle ne recoupe pas nos montants : on l'utilise pour les **tendances**, pas pour les totaux.

**Qu'est-ce qu'un client ?** Une entreprise enregistrée qui passe quelques commandes d'essai n'est pas encore un client. On compte une entreprise comme **cliente active** une année si elle y passe **au moins 10 commandes ou 100 000 $ d'achats**. Un **nouveau client** est une entreprise active pour la première fois en 2020 ou en 2021.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Entreprises actives par année
# ---------------------------------------------------------------------------
en_ligne_detail = pd.read_parquet(config.CLEAN / "fait_ventes_en_ligne.parquet",
                                  columns=["annee", "id_client", "id_produit", "numero_commande", "montant_ventes", "type_client"])
clients_dim = pd.read_csv(config.DIM / "DimCustomer.csv", encoding="utf-8-sig", usecols=["CustomerKey", "GeographyKey"])
geographie = pd.read_csv(config.DIM / "DimGeography.csv", encoding="utf-8-sig", usecols=["GeographyKey", "RegionCountryName"])
pays_client = clients_dim.merge(geographie, on="GeographyKey", how="left").set_index("CustomerKey").RegionCountryName.str.strip()

entreprises = en_ligne_detail[en_ligne_detail.type_client == "Entreprise"].copy()
entreprises["pays"] = entreprises.id_client.map(pays_client)
entreprises["nouveaute"] = entreprises.annee == entreprises.id_produit.map(lancement)

SEUIL_COMMANDES, SEUIL_MONTANT = 10, 100_000
activite = entreprises.groupby(["id_client", "annee"]).agg(commandes=("numero_commande", "nunique"), achats=("montant_ventes", "sum"))
activite["active"] = (activite.commandes >= SEUIL_COMMANDES) | (activite.achats >= SEUIL_MONTANT)
print("Entreprises enregistrées et actives, par année :")
display(activite.groupby("annee").agg(entreprises_avec_achat=("active", "size"), actives=("active", "sum"),
                                      achats_des_non_actives_maximum=("achats", lambda s: s[~activite.loc[s.index, "active"]].max()))
        .style.format({"achats_des_non_actives_maximum": "{:,.0f}"}, na_rep="(aucune)"))

# ---------------------------------------------------------------------------
# 2. Statut de chaque entreprise : historique (active en 2019) ou nouvelle (active pour la première fois ensuite)
# ---------------------------------------------------------------------------
actives = activite[activite.active].reset_index()
premiere_annee = actives.groupby("id_client").annee.min()
statut = premiere_annee.map(lambda a: "historique (active en " + str(REFERENCE) + ")" if a == REFERENCE else f"nouvelle ({a})")
entreprises["statut"] = entreprises.id_client.map(statut).fillna("jamais active")
achats_statut = entreprises.pivot_table(index="statut", columns="annee", values="montant_ventes", aggfunc="sum", fill_value=0) / 1e6
achats_statut["entreprises"] = entreprises.groupby("statut").id_client.nunique()
achats_statut.loc["Total"] = achats_statut.sum()
print("Achats des entreprises selon leur statut (M$) :")
display(achats_statut.style.format({a: "{:,.1f}" for a in ANNEES} | {"entreprises": "{:.0f}"}))
part_nouvelles = (achats_statut.loc[achats_statut.index.str.startswith("nouvelle"), EXERCICE].sum() * 1e6
                  / en_ligne_detail.loc[en_ligne_detail.annee == EXERCICE, "montant_ventes"].sum())
print(f"Part des nouvelles entreprises dans le détail des ventes en ligne en {EXERCICE} : {part_nouvelles:.0%}")

# Les historiques : combien sont encore actives en fin de période ?
historiques = premiere_annee[premiere_annee == REFERENCE].index
encore_actives = actives[(actives.annee == EXERCICE) & actives.id_client.isin(historiques)].id_client.nunique()
print(f"Entreprises historiques : {len(historiques)} · encore actives en {EXERCICE} : {encore_actives}")

# ---------------------------------------------------------------------------
# 3. Par pays
# ---------------------------------------------------------------------------
par_pays = entreprises.pivot_table(index=["statut", "pays"], columns="annee", values="montant_ventes", aggfunc="sum", fill_value=0) / 1e6
par_pays = par_pays[par_pays.sum(axis=1) > 0.05].sort_values(EXERCICE, ascending=False)
print("Achats des entreprises par statut et par pays (M$) :")
display(par_pays.style.format("{:,.1f}"))

# ---------------------------------------------------------------------------
# 4. Part des nouveautés de l'année dans les achats, selon le statut
# ---------------------------------------------------------------------------
part_nouveautes = entreprises.groupby(["statut", "annee"]).apply(
    lambda d: d.loc[d.nouveaute, "montant_ventes"].sum() / d.montant_ventes.sum() if d.montant_ventes.sum() > 1e5 else np.nan,
    include_groups=False).unstack()
print("Part des nouveautés de l'année dans les achats (années avec plus de 100 000 $ d'achats) :")
display(part_nouveautes.style.format("{:.1%}", na_rep=""))

h = part_nouveautes.loc[part_nouveautes.index.str.startswith("historique")].iloc[0]
nouvelles_exercice = entreprises[entreprises.statut.str.startswith("nouvelle") & (entreprises.annee == EXERCICE)]
part_nouveautes_nouvelles = nouvelles_exercice.loc[nouvelles_exercice.nouveaute, "montant_ventes"].sum() / nouvelles_exercice.montant_ventes.sum()
verdict("Clients en ligne", "Les entreprises clientes se renouvellent-elles d'une année sur l'autre ?",
        f"{achats_statut.loc['nouvelle (' + str(EXERCICE - 1) + ')', EXERCICE - 1]:,.1f} M$ d'achats des nouvelles de {EXERCICE - 1} cette année-là, "
        f"{achats_statut.loc['nouvelle (' + str(EXERCICE - 1) + ')', EXERCICE]:,.1f} M$ l'année suivante".replace(".", ","), "retenue")
verdict("Clients en ligne", "Les grands comptes historiques restent-ils ?",
        f"{encore_actives} sur {len(historiques)} encore actives en {EXERCICE}", "retenue")
verdict("Clients en ligne", f"Part des nouvelles entreprises dans le détail des ventes en ligne en {EXERCICE}", f"{part_nouvelles:.0%}", "retenue")
verdict("Clients en ligne", "Les comptes historiques étaient-ils de gros acheteurs de nouveautés ?",
        f"{h[REFERENCE]:.0%} de nouveautés en {REFERENCE}, {h[REFERENCE + 1]:.0%} l'année suivante ; nouvelles entreprises : {part_nouveautes_nouvelles:.0%} en {EXERCICE}",
        "retenue")
verdict("Clients en ligne", "Pourquoi les comptes partent : une entreprise veut un fournisseur qui suit tout son parc",
        "interprétation métier, cohérente avec le départ en bloc des comptes quand leurs achats de nouveautés s'effondrent", "indice")

**Lecture**

- **La définition du client change la lecture.** En 2019, 43 entreprises achètent, mais 16 seulement sont actives : les 27 autres passent quelques commandes d'essai, 3 778 $ au plus sur l'année. En 2020 et 2021, 28 entreprises sont actives chaque année.
- **Les grands comptes historiques partent.** Les 16 entreprises actives en 2019 achetaient 169,0 M$. Elles n'en achètent plus que 86,6 M$ en 2020, et 0,1 M$ en 2021 : une seule est encore active.
- **Les nouveaux clients font toute la boutique en ligne en 2021**, soit 91 % du détail des ventes en ligne. Mais ils se renouvellent eux aussi : les 13 entreprises devenues clientes en 2020 achètent 149,4 M$ cette année-là, puis 49,0 M$ en 2021. Celles de 2021 (14 entreprises, 177,4 M$) les remplacent, y compris au Royaume-Uni (53,6 M$), en Allemagne (24,5 M$) et en Russie (23,6 M$).
- **Les nouveautés avant chaque départ.** Les comptes historiques consacraient 40,5 % de leurs achats aux nouveautés de l'année en 2019. Cette part tombe à 17,4 % en 2020, puis ils partent. Les clients de 2020 suivent le même chemin : 22,8 % de nouveautés en 2020, 4,7 % en 2021, et des achats divisés par trois. En 2021, les nouvelles entreprises n'achètent que 10 % de nouveautés.

**Interprétation métier** (présentée comme telle) : une entreprise cherche un fournisseur qui suit l'ensemble de son parc. Quand l'offre de nouveautés s'épuise, elle ne réduit pas ses achats, elle part en bloc chez un fournisseur qui suit. C'est ce « tout ou rien » qu'on observe : les comptes ne déclinent pas doucement, ils disparaissent.

**Verdict** : les constats (départ des comptes historiques, poids des nouveaux clients, effondrement de la part des nouveautés avant chaque départ) sont *retenus*. L'explication par le besoin d'un fournisseur qui suit tout le parc est une *interprétation métier*, cohérente avec ces constats. Elle rejoint le reste de l'analyse : **sans nouveautés, on ne garde pas les clients professionnels**.

## 8. Une seule stratégie : renouveler l'assortiment

On ne peut pas dire, couple par couple, si un produit non vendu était en rayon. Mais il n'est pas nécessaire de le savoir pour recommander, parce que les deux branches conduisent à la même stratégie :

| Branche | Ce qu'elle dit | Ce qu'il faut faire |
|---|---|---|
| A. Stock mort | Le magasin porte des produits qui ne tournent plus | **Libérer ce stock** et réinvestir dans de nouvelles références. Cette branche renforce H1 : la place et l'argent immobilisés manquent aux nouveautés |
| B. Assortiment réduit | Le produit, ou sa nouvelle version, n'est pas en rayon | **Mettre les nouveautés en rayon**. Un article absent ne se vend pas, quelle que soit la demande |

Les deux branches convergent vers **le renouvellement de l'assortiment des magasins** : moins de produits qui dorment, plus de produits récents en rayon. Elles ne s'opposent pas, elles se complètent.

**Ce que chiffre `06_q1_facteurs_rentabilite` (section 8), sur l'exercice 2021**

| Volet | Comment | Hypothèse à signaler |
|---|---|---|
| Revenir au rythme de lancement de 2019 | Nouveautés supplémentaires × chiffre d'affaires d'une nouveauté de l'année en 2021 × ce que rapporte 1 $ de vente | Gain brut : le coût de lancement n'est pas dans les données |
| Remettre les nouveautés en rayon | Ventes des nouveautés en magasin si elles s'y vendaient, par rapport à la boutique en ligne, comme les produits anciens | Le magasin vend une nouveauté aussi bien qu'un produit ancien |
| Libérer le stock dormant | Couples (magasin, produit) sans vente × stock moyen d'un produit par relevé × coût unitaire ; par transfert vers la boutique en ligne, retour fournisseur ou déstockage, **pas par la remise** (`09_q5_remises`) | Le stock n'est pas relevé partout : valeur estimée ; taux de récupération à faire varier ; trésorerie récupérée une fois, pas un gain annuel |

Cette stratégie s'ajoute aux autres leviers déjà chiffrés (charges, publicité, remises), sur la même base.

## 9. Synthèse

In [ ]:
registre = pd.DataFrame(verdicts)
pd.set_option("display.max_colwidth", 160)
display(registre.style.hide(axis="index"))

chemin = config.L4 / "offre_tableaux.xlsx"
with pd.ExcelWriter(chemin) as classeur:
    registre.to_excel(classeur, sheet_name="Verdicts", index=False)
    canal.to_excel(classeur, sheet_name="Canaux")
    par_magasin.to_excel(classeur, sheet_name="Evolution_magasins")
    par_categorie.to_excel(classeur, sheet_name="Categories")
    lancements.to_excel(classeur, sheet_name="Lancements")
    age.to_excel(classeur, sheet_name="Age_des_produits")
    decomposition.to_excel(classeur, sheet_name="Decomposition_magasins")
    cohorte.to_excel(classeur, sheet_name="Memes_produits")
    indice1.to_excel(classeur, sheet_name="Indice1_rotation")
    indice2.to_excel(classeur, sheet_name="Indice2_dispersion")
    indice3.to_excel(classeur, sheet_name="Indice3_pays")
    indice4.to_excel(classeur, sheet_name="Indice4_retour")
    tableau_gammes.to_excel(classeur, sheet_name="Gammes_de_prix")
    tous.to_excel(classeur, sheet_name="Entree_de_gamme")
    memes_produits.to_excel(classeur, sheet_name="Entree_de_gamme_memes_prod")
    chronologie.to_excel(classeur, sheet_name="Entree_de_gamme_trimestres")
    achats_statut.to_excel(classeur, sheet_name="Clients_statut")
    par_pays.to_excel(classeur, sheet_name="Clients_pays")
    part_nouveautes.to_excel(classeur, sheet_name="Clients_nouveautes")
print("Tableaux enregistrés :", chemin.relative_to(config.ROOT))

### Réponse : pourquoi les ventes des magasins ont chuté

**Le fil conducteur : un assortiment qui vieillit faute de nouveautés.** On le retrouve à quatre endroits indépendants.

1. **La chute est générale et porte sur les volumes** : tous les magasins, tous les mois, toutes les catégories. Les unités vendues en magasin baissent de 37,5 % pendant qu'elles montent de 60 % en ligne. En magasin, ce sont les produits à plus de 100 $ qui s'effondrent.
2. **Le catalogue se renouvelle moins** (*retenue*) : 552 lancements en 2019, 272 en 2021, aucun au second semestre 2021. C'est la plus grosse composante de la chute (69 % dans la décomposition comptable), mais elle touche aussi la boutique en ligne.
3. **Les clients descendent en gamme** (*retenue*), dans tous les canaux et sur les mêmes produits. Les restrictions COVID ne l'expliquent pas (*non soutenue*). Un arbitrage des ménages pendant la crise est compatible avec les chiffres officiels (*ouverte*).
4. **Chaque produit se vend dans deux fois moins de magasins** (*retenue*), et c'est ce qui sépare les magasins de la boutique en ligne. Là où il se vend, un produit recule comme en ligne.
5. **Un produit non vendu dans un magasin est plus probablement en rayon sans tourner** (stock mort, *indice*) **que retiré** (assortiment réduit, *non soutenue*). On ne peut pas le prouver couple par couple.
6. **Les grands comptes en ligne partent quand les nouveautés s'épuisent** (*retenue*, avec une explication métier), et la boutique en ligne ne tient que par le renouvellement de sa clientèle professionnelle.
7. **Ce qui reste ouvert** : pourquoi les magasins ne compensent pas la baisse des nouveautés avec le reste du catalogue, comme le fait la boutique en ligne grâce à ses achats en gros.

### Recommandation

**Renouveler l'assortiment des magasins**, la même action quelle que soit la branche :

- si c'est du stock mort, on le libère et on réinvestit dans des nouveautés (sans passer par la remise, qui ne fait pas bouger les volumes : `09_q5_remises`) ;
- si c'est un assortiment réduit, on remet les nouveautés en rayon, puisqu'un produit absent ne se vend pas ;
- dans les deux cas, c'est aussi la condition pour garder les clients professionnels de la boutique en ligne, qui partent quand l'offre ne suit plus leur parc.

Les montants (stock à libérer, ventes attendues des nouveautés, variantes avec transfert de la longue traîne en ligne) sont chiffrés dans `06_q1_facteurs_rentabilite` (section 8), avec les autres leviers de l'exercice 2021.

### Limites

- La table de stock est un relevé partiel, lié aux ventes : elle ne mesure ni le stock mort ni les ruptures.
- La décomposition par âge des produits est comptable : elle dit où est la baisse, pas ce qui l'a causée.
- La partie sur les entreprises clientes repose sur la table des ventes en ligne détaillées, qui ne recoupe pas nos montants : ses chiffres ne s'additionnent pas avec ceux des autres sections.
- Le contexte économique (consommation, épargne, inflation) vient des statistiques d'Eurostat, pas de nos données.